# Furiosa Holter 추가 진단

`std3` 서버에서 기존 356명 5-fold 결과와 추가 입력·사례 및 분리 임계값 실험을 읽습니다. 예측 값은 연구용이며 임상 판단에 사용할 수 없습니다. 이 노트북은 기존 실험을 다시 실행하지 않습니다.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("/root/.cache/std3_holter/llm_compare")
for title, name in [
    ("기존 5-fold", "run_20261102_cv5"),
    ("입력 및 사례 ablation", "run_20261102_diagnostics"),
    ("분리 보정 임계값", "run_20261102_threshold"),
    ("흡연 변수 및 임상 전용 문구", "run_20261102_smoking"),
]:
    folder = ROOT / name
    print("\n", title)
    if not (folder / "metadata.json").exists():
        print("아직 시작되지 않았습니다.")
        continue
    metadata = json.loads((folder / "metadata.json").read_text())
    print("상태:", metadata["status"])
    if (folder / "predictions.csv").exists():
        frame = pd.read_csv(folder / "predictions.csv")
        if "condition" in frame.columns:
            display(frame.groupby(["condition", "status"]).size().rename("건수").reset_index())
        else:
            display(frame.groupby(["role", "status"]).size().rename("건수").reset_index())
    if (folder / "metrics.json").exists():
        metrics = json.loads((folder / "metrics.json").read_text())
        if name == "run_20261102_threshold":
            print("보정 fold와 분리된 평가 결과:", metrics["pooled"])
            display(pd.DataFrame(metrics["folds"]))
        else:
            display(pd.DataFrame({k: v["all_successful"] for k, v in metrics.items()}).T)